# LeetCode 1702 — Maximum Binary String After Change bằng Python

Link đề: https://leetcode.com/problems/maximum-binary-string-after-change/ · Độ khó: **Medium** · Chủ đề: String, Greedy

Skeleton LeetCode:

```python
class Solution:
    def maximumBinaryString(self, binary: str) -> str:
        ...
```

Lộ trình: **BFS trên mọi chuỗi đạt được** (vét cạn, `n ≤ 12`) → quan sát kết quả để đoán quy luật → **mô phỏng tham lam** từ trái sang phải (`O(n²)`) → **công thức đóng** `O(n)`: đáp án có **nhiều nhất một số 0**, nằm ở vị trí `(chỉ số số 0 đầu tiên) + (số lượng số 0) − 1`.

## 1. Mục tiêu học

1. Hiểu hai phép biến đổi: `"00" → "10"` (biến một số 0 thành 1) và `"10" → "01"` (đẩy số 0 sang **trái**, số 1 sang phải).
2. Dùng **BFS trên không gian trạng thái** (tập chuỗi) để tìm đáp án chuẩn và **quan sát quy luật** từ dữ liệu.
3. Tìm **bất biến**: số 0 không bao giờ tăng; phần `1` ở đầu chuỗi (trước số 0 đầu tiên) không bao giờ đổi.
4. Chứng minh tham lam và rút công thức `O(n)` (đếm số 0 + vị trí số 0 đầu).
5. So sánh chuỗi nhị phân cùng độ dài theo giá trị bằng **so sánh từ điển**.

## 2. Đề bài & ví dụ

Cho chuỗi nhị phân `binary` (chỉ gồm `'0'` và `'1'`). Có thể áp dụng các phép sau **bao nhiêu lần tùy ý**:
- **Phép 1:** nếu có chuỗi con `"00"`, thay nó bằng `"10"`. Ví dụ `"00010" → "10010"`.
- **Phép 2:** nếu có chuỗi con `"10"`, thay nó bằng `"01"`. Ví dụ `"00010" → "00001"`.

Trả về chuỗi nhị phân **lớn nhất** (theo giá trị thập phân) có thể nhận được.

**Ví dụ 1:** `binary = "000110"` → `"111011"`. Một dãy biến đổi:
`"000110" → "000101" → "100101" → "110101" → "110011" → "111011"`.

**Ví dụ 2:** `binary = "01"` → `"01"` (không biến đổi được gì).

**Ràng buộc:**
- `1 ≤ binary.length ≤ 10⁵`
- `binary` chỉ gồm `'0'` và `'1'`.

In [1]:
import random
import time
from collections import deque
from typing import List

examples = [
    ("000110", "111011"),
    ("01", "01"),
]
for b, want in examples:
    print(f"binary={b!r} -> kỳ vọng {want!r}")

binary='000110' -> kỳ vọng '111011'
binary='01' -> kỳ vọng '01'


## 3. Kiến thức nền

### 3.1 So sánh chuỗi nhị phân cùng độ dài

Hai phép biến đổi **giữ nguyên độ dài**. Với hai chuỗi nhị phân cùng độ dài, so sánh giá trị ⇔ so sánh từ điển (`'0' < '1'`). Nên `max(list_of_strings)` cho đúng chuỗi lớn nhất.

In [2]:
a, b = "110011", "101111"
print(a > b, int(a, 2) > int(b, 2))

True True


### 3.2 Thay thế tại một vị trí cụ thể

`s.replace("00", "10")` thay **mọi** lần xuất hiện (từ trái, không chồng lấn) — không phải "một lần áp dụng phép". Để áp phép đúng một lần tại vị trí `i`, dùng slicing: `s[:i] + "10" + s[i+2:]`.

In [3]:
s = "0000"
print("replace toàn bộ:", s.replace("00", "10"))
print("một lần tại i=1:", s[:1] + "10" + s[3:])

replace toàn bộ: 1010
một lần tại i=1: 0100


### 3.3 BFS trên tập trạng thái

Mỗi chuỗi là một đỉnh; mỗi phép áp dụng tại một vị trí là một cạnh. BFS từ chuỗi ban đầu liệt kê **mọi** chuỗi đạt được. Dùng `set` để không thăm lại, `deque` làm hàng đợi.

In [4]:
def neighbors(s):
    for i in range(len(s) - 1):
        if s[i:i + 2] == "00":
            yield s[:i] + "10" + s[i + 2:]
        elif s[i:i + 2] == "10":
            yield s[:i] + "01" + s[i + 2:]

print(list(neighbors("0010")))

['1010', '0001']


### 3.4 `str.find` và `str.count`

`s.find('0')` trả về chỉ số `'0'` đầu tiên (hoặc `-1`), `s.count('0')` đếm số `'0'` — cả hai chạy bằng C, `O(n)`.

In [5]:
s = "1100101"
print(s.find("0"), s.count("0"), "1111".find("0"))

2 3 -1


## 4. Ý tưởng nền tảng / trực giác

Hãy xem phép biến đổi làm gì:
- **Phép 1 `00 → 10`:** giảm số lượng `0` đi một, và biến `0` **bên trái** thành `1` — rất có lợi.
- **Phép 2 `10 → 01`:** một `0` nhảy sang **trái** một ô, đổi chỗ với `1`. Bản thân nó làm chuỗi **nhỏ đi**, nhưng nó giúp **gom các số 0 lại gần nhau** để dùng Phép 1.

Thử tay `"0101"`:
- `0 1 0 1` — `0` thứ hai (ở vị trí 2) nhảy trái qua `1`: `"0011"`.
- `00 → 10`: `"1011"`.
- Chỉ còn một `0`, không thể bỏ nó ⇒ đáp án `"1011"`.

Quan sát: **cuối cùng còn đúng một `0`** (nếu ban đầu có ít nhất một `0`), và nó nằm ở vị trí `(số 0 đầu tiên) + (số lượng 0) − 1`. Với `"0101"`: số 0 đầu ở vị trí 0, có 2 số 0 ⇒ vị trí `1` ⇒ `"1011"`. ✓

In [6]:
def bfs_all(s):
    seen = {s}; q = deque([s])
    while q:
        u = q.popleft()
        for v in neighbors(u):
            if v not in seen:
                seen.add(v); q.append(v)
    return seen

reach = bfs_all("0101")
print("số chuỗi đạt được:", len(reach))
print("lớn nhất:", max(reach))
print("mọi chuỗi:", sorted(reach, reverse=True))

số chuỗi đạt được: 4
lớn nhất: 1011
mọi chuỗi: ['1011', '0111', '0101', '0011']


## 5. Cách 1 — Brute Force / baseline: BFS mọi chuỗi đạt được

**Ý tưởng:** BFS như trên, trả về `max(seen)`. Đúng tuyệt đối nhưng số chuỗi đạt được có thể lên tới `2ⁿ`.

In [7]:
def max_binary_bfs(s):
    return max(bfs_all(s))

for b, want in examples:
    print(max_binary_bfs(b), "kỳ vọng", want)

111011 kỳ vọng 111011
01 kỳ vọng 01


**Độ phức tạp:** tối đa `2ⁿ` trạng thái, mỗi trạng thái `O(n)` láng giềng, mỗi láng giềng tạo chuỗi `O(n)` ⇒ `O(2ⁿ · n²)`. Chỉ dùng được với `n ≤ ~14`.

Dùng BFS để **thu thập dữ liệu** và đoán quy luật:

In [8]:
for s in ["000", "0011", "0110", "1010", "11001", "100100", "0000111", "110"]:
    best = max_binary_bfs(s)
    print(f"{s:>8} -> {best:>8} | số 0 ban đầu = {s.count('0')}, số 0 đầu ở {s.find('0')}, "
          f"số 0 trong đáp án = {best.count('0')}, vị trí = {best.find('0')}")

     000 ->      110 | số 0 ban đầu = 3, số 0 đầu ở 0, số 0 trong đáp án = 1, vị trí = 2
    0011 ->     1011 | số 0 ban đầu = 2, số 0 đầu ở 0, số 0 trong đáp án = 1, vị trí = 1
    0110 ->     1011 | số 0 ban đầu = 2, số 0 đầu ở 0, số 0 trong đáp án = 1, vị trí = 1
    1010 ->     1101 | số 0 ban đầu = 2, số 0 đầu ở 1, số 0 trong đáp án = 1, vị trí = 2
   11001 ->    11101 | số 0 ban đầu = 2, số 0 đầu ở 2, số 0 trong đáp án = 1, vị trí = 3
  100100 ->   111101 | số 0 ban đầu = 4, số 0 đầu ở 1, số 0 trong đáp án = 1, vị trí = 4
 0000111 ->  1110111 | số 0 ban đầu = 4, số 0 đầu ở 0, số 0 trong đáp án = 1, vị trí = 3
     110 ->      110 | số 0 ban đầu = 1, số 0 đầu ở 2, số 0 trong đáp án = 1, vị trí = 2


In [9]:
random.seed(0)
for n in [10, 12, 14]:
    s = "".join(random.choice("01") for _ in range(n))
    t0 = time.perf_counter(); r = max_binary_bfs(s)
    print(f"n={n}: {s} -> {r}  ({len(bfs_all(s))} trạng thái, {time.perf_counter() - t0:.3f}s)")

n=10: 1101111110 -> 1110111111  (28 trạng thái, 0.000s)
n=12: 010010100110 -> 111111011111  (1496 trạng thái, 0.005s)
n=14: 11101110001011 -> 11111110111111  (1319 trạng thái, 0.005s)


## 6. Ý tưởng thuật toán / bước đột phá

**Bất biến 1 — tiền tố toàn `1` không đổi được và không cần đổi.** Gọi `k` = chỉ số số `0` đầu tiên. Các ký tự `s[0..k−1]` đều là `1`. Phép 2 kéo `0` sang trái — có thể làm phần đầu xấu đi, nhưng ta không bao giờ cần làm vậy: phần đầu đã là `1`, tối ưu rồi. Nếu không có `0` nào, đáp án là chính `s`.

**Bất biến 2 — số `0` không tăng.** Phép 1 giảm 1, phép 2 giữ nguyên. Mỗi lần Phép 1 cần **hai** số 0 cạnh nhau ⇒ nếu có `z ≥ 1` số 0, ta không thể xóa hết: luôn còn **ít nhất một** `0`.

**Bất biến 3 — số 0 còn lại không thể tiến quá xa.** Mỗi lần phép 1 xóa một số 0 tại vị trí số 0 đầu tiên, số 0 đầu tiên tiến sang phải đúng **1 ô**; phép 2 chỉ kéo số 0 sang **trái**. Với `z` số 0, ta xóa được tối đa `z − 1` lần ⇒ số 0 còn lại ở vị trí tối đa `k + z − 1` (chứng minh chặt ở mục 10 bằng đại lượng `g`).

**Cấu trúc đáp án (khi `z ≥ 1`):** `"1" * (k + z − 1) + "0" + "1" * (n − k − z)`.

**Quy tắc tham lam:** duyệt trái → phải; tại mỗi `0` ở vị trí `i` (không phải số 0 cuối cùng), kéo số `0` kế tiếp về `i+1` bằng phép 2, rồi dùng phép 1 biến `s[i]` thành `1`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Mô phỏng tham lam bằng mảng ký tự (`O(n²)`)

Từng đoạn:
1. `a = list(s)`, `i = s.find('0')`.
2. Trong khi còn số `0` ở sau `i`: tìm `j > i` là số `0` kế tiếp. Dùng phép 2 lặp lại để đưa nó về `i+1` (thực chất: `a[j] = '1'`, `a[i+1] = '0'` sau một loạt đổi chỗ — các `1` giữa `i+1..j−1` bị đẩy sang phải 1 ô, nhưng chúng đều là `1` nên kết quả như hoán đổi `a[i+1]` và `a[j]`).
3. Áp phép 1 tại `i`: `a[i] = '1'`, số 0 giờ ở `i+1`. Gán `i = i + 1`.

Mô phỏng từng phép 2 một (không dùng hoán đổi tắt) cho độ phức tạp `O(n²)` — dùng để **minh họa** và kiểm chứng.

In [10]:
def max_binary_simulate(s):
    a = list(s)
    n = len(a)
    i = s.find("0")
    if i == -1:
        return s
    ops = 0
    while True:
        j = i + 1
        while j < n and a[j] == "1":
            j += 1
        if j == n:                       # không còn số 0 nào sau i
            break
        while j > i + 1:                 # phép 2 liên tiếp: "10" -> "01" đưa số 0 từ j về i+1
            a[j - 1], a[j] = "0", "1"
            j -= 1; ops += 1
        a[i] = "1"                       # phép 1: "00" -> "10" tại i
        ops += 1
        i += 1
    return "".join(a)

for b, want in examples:
    print(max_binary_simulate(b), "kỳ vọng", want)

111011 kỳ vọng 111011
01 kỳ vọng 01


**Độ phức tạp:** mỗi số 0 có thể phải đi qua `O(n)` số 1 ⇒ Time `O(n²)` worst case, Space `O(n)`.

### 7.2 Cách 3 — Mô phỏng tham lam với hoán đổi tắt (`O(n)`)

Nhận xét ở bước 2: loạt phép 2 chỉ để đưa số 0 ở `j` về `i+1`; kết quả bằng hoán đổi `a[i+1]` ↔ `a[j]`. Con trỏ `j` chỉ tiến về phải ⇒ `O(n)`.

In [11]:
def max_binary_two_pointers(s):
    a = list(s)
    n = len(a)
    i = s.find("0")
    if i == -1:
        return s
    j = i + 1
    while True:
        while j < n and a[j] == "1":
            j += 1
        if j == n:
            break
        a[j], a[i + 1] = "1", "0"        # gom số 0 kế tiếp về cạnh i
        a[i] = "1"                       # 00 -> 10
        i += 1
        j = max(j, i + 1)
    return "".join(a)

for b, want in examples:
    print(max_binary_two_pointers(b), "kỳ vọng", want)

111011 kỳ vọng 111011
01 kỳ vọng 01


**Độ phức tạp:** `i` và `j` đều chỉ tăng ⇒ Time `O(n)`, Space `O(n)`.

### 7.3 Cách 4 — Công thức đóng

Từng đoạn:
1. `k = s.find('0')`; nếu `k == -1` trả về `s`.
2. `z = s.count('0')`.
3. Trả về `'1' * (k + z − 1) + '0' + '1' * (n − k − z)`.

In [12]:
def max_binary_formula(s):
    k = s.find("0")
    if k == -1:
        return s
    z = s.count("0")
    n = len(s)
    return "1" * (k + z - 1) + "0" + "1" * (n - k - z)

for b, want in examples:
    print(max_binary_formula(b), "kỳ vọng", want)

111011 kỳ vọng 111011
01 kỳ vọng 01


**Độ phức tạp:** Time `O(n)` (find + count + tạo chuỗi), Space `O(n)` cho chuỗi kết quả.

## 8. TRACE từng bước

Trace Cách 2 (mô phỏng từng phép) trên `"000110"`:

In [13]:
def trace_simulate(s):
    a = list(s); n = len(a); i = s.find("0")
    print(f"bắt đầu: {''.join(a)}  (số 0 đầu ở {i})")
    if i == -1:
        return s
    while True:
        j = i + 1
        while j < n and a[j] == "1":
            j += 1
        if j == n:
            print(f"không còn số 0 sau vị trí {i} -> dừng"); break
        while j > i + 1:
            a[j - 1], a[j] = "0", "1"; j -= 1
            print(f"  phép 2 (10->01) tại {j}:   {''.join(a)}")
        a[i] = "1"
        print(f"  phép 1 (00->10) tại {i}:   {''.join(a)}")
        i += 1
    return "".join(a)

trace_simulate("000110")

bắt đầu: 000110  (số 0 đầu ở 0)
  phép 1 (00->10) tại 0:   100110
  phép 1 (00->10) tại 1:   110110
  phép 2 (10->01) tại 4:   110101
  phép 2 (10->01) tại 3:   110011
  phép 1 (00->10) tại 2:   111011
không còn số 0 sau vị trí 3 -> dừng


'111011'

In [14]:
trace_simulate("1101001")

bắt đầu: 1101001  (số 0 đầu ở 2)
  phép 2 (10->01) tại 3:   1100101
  phép 1 (00->10) tại 2:   1110101
  phép 2 (10->01) tại 4:   1110011
  phép 1 (00->10) tại 3:   1111011
không còn số 0 sau vị trí 4 -> dừng


'1111011'

Bảng cho Cách 4 trên vài input:

| binary | k (số 0 đầu) | z (số lượng 0) | vị trí 0 cuối = k+z−1 | đáp án |
|---|---|---|---|---|
| `000110` | 0 | 4 | 3 | `111011` |
| `01` | 0 | 1 | 0 | `01` |
| `1101001` | 2 | 3 | 4 | `1111011` |
| `1111` | −1 | 0 | — | `1111` |

In [15]:
def trace_formula(s):
    k = s.find("0"); z = s.count("0")
    if k == -1:
        print(f"{s}: không có số 0 -> giữ nguyên"); return s
    res = max_binary_formula(s)
    print(f"{s}: k={k}, z={z}, vị trí 0 cuối = {k + z - 1} -> {res}")
    return res

for s in ["000110", "01", "1101001", "1111"]:
    trace_formula(s)

000110: k=0, z=4, vị trí 0 cuối = 3 -> 111011
01: k=0, z=1, vị trí 0 cuối = 0 -> 01
1101001: k=2, z=3, vị trí 0 cuối = 4 -> 1111011
1111: không có số 0 -> giữ nguyên


## 9. Code nộp LeetCode

In [16]:
class Solution:
    def maximumBinaryString(self, binary: str) -> str:
        k = binary.find("0")
        if k == -1:
            return binary
        z = binary.count("0")
        n = len(binary)
        return "1" * (k + z - 1) + "0" + "1" * (n - k - z)

sol = Solution()
for b, want in examples:
    got = sol.maximumBinaryString(b)
    print(got, "OK" if got == want else "SAI")

111011 OK
01 OK


## 10. Vì sao thuật toán đúng

Giả sử `z ≥ 1`, `k` = chỉ số số 0 đầu tiên. Gọi `R = "1"·(k+z−1) + "0" + "1"·(n−k−z)`.

**(a) R đạt được.** Cách 2 tạo ra đúng `R`: mỗi vòng biến vị trí `i` thành `1` và đặt số 0 ở `i+1`, đồng thời "tiêu thụ" một số 0 phía sau. Sau `z − 1` vòng, số 0 duy nhất ở `k + z − 1`, mọi ô khác là `1`. Các phép dùng đều hợp lệ (phép 2 chỉ áp trên `"10"`, phép 1 trên `"00"`).

**(b) Không thể tốt hơn R.** Định nghĩa đại lượng

`g(s) = (chỉ số số 0 đầu tiên của s) + (số lượng số 0 trong s)`.

Kiểm tra từng phép:
- **Phép 2** `10 → 01`: số lượng 0 giữ nguyên; số 0 đầu tiên chỉ có thể **lùi sang trái** (nếu chính nó nhảy) hoặc giữ nguyên ⇒ `g` không tăng.
- **Phép 1** `00 → 10` **tại** số 0 đầu tiên: chỉ số số 0 đầu tăng đúng 1, số lượng giảm 1 ⇒ `g` không đổi.
- **Phép 1** ở chỗ khác: số 0 đầu giữ nguyên, số lượng giảm 1 ⇒ `g` giảm.

Vậy `g(s') ≤ g(s) = k + z` với mọi chuỗi `s'` đạt được. Ngoài ra phép 1 cần hai số 0 và chỉ xóa một ⇒ `s'` luôn còn `z' ≥ 1` số 0, nên số 0 đầu của `s'` ở vị trí `k' ≤ k + z − z' ≤ k + z − 1`. Trong khi đó `R` có toàn `1` trên `[0, k+z−2]` ⇒ ở vị trí `k'` (hoặc sớm hơn) `s'` có `0` còn `R` có `1`, trừ khi `k' = k+z−1` và khi đó `R` là toàn `1` ở mọi vị trí khác ⇒ `s' ≤ R`. ∎

(Trường hợp `z = 0`: không phép nào áp được ⇒ đáp án là `s`.)

In [17]:
# kiểm chứng bất biến g(s) = (vị trí 0 đầu) + (số lượng 0) không tăng trên mọi cạnh BFS
def g(s):
    return s.find("0") + s.count("0")

random.seed(3)
cnt = 0
for _ in range(200):
    s = "".join(random.choice("01") for _ in range(random.randint(1, 10)))
    if "0" not in s:
        continue
    for u in bfs_all(s):
        for v in neighbors(u):
            assert g(v) <= g(u)
            cnt += 1
print("đã kiểm", cnt, "cạnh: g không bao giờ tăng")

đã kiểm 49200 cạnh: g không bao giờ tăng


## 11. Phân tích Time / Space

Gọi `n = len(binary) ≤ 10⁵`, `z` = số lượng số 0.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. BFS | `O(2ⁿ · n²)` | `O(2ⁿ · n)` | `n ≤ 14` |
| 2. Mô phỏng từng phép | `O(n²)` worst | `O(n)` | minh họa |
| 3. Hai con trỏ | `O(n)` | `O(n)` | mô phỏng nhanh |
| 4. Công thức | `O(n)` | `O(n)` (kết quả) | **nộp bài** |

In [18]:
random.seed(4)
s_big = "".join(random.choice("01") for _ in range(10**5))
s_bad = "0" + "1" * 2000 + "0" * 2000     # worst case cho mô phỏng từng phép
for name, fn in [("simulate", max_binary_simulate), ("2ptr", max_binary_two_pointers), ("formula", max_binary_formula)]:
    t0 = time.perf_counter(); r1 = fn(s_bad); t1 = time.perf_counter(); r2 = fn(s_big) if name != "simulate" else None
    t2 = time.perf_counter()
    print(f"{name:8s}: chuỗi xấu n={len(s_bad)} {t1 - t0:.3f}s" + (f" | n=1e5 {t2 - t1:.3f}s" if r2 else ""))

simulate: chuỗi xấu n=4001 0.339s
2ptr    : chuỗi xấu n=4001 0.000s | n=1e5 0.008s
formula : chuỗi xấu n=4001 0.000s | n=1e5 0.000s


## 12. Bộ test

In [19]:
fns = [max_binary_simulate, max_binary_two_pointers, max_binary_formula,
       lambda b: Solution().maximumBinaryString(b)]

for b, want in examples:
    for fn in fns:
        assert fn(b) == want

edge = [("0", "0"), ("1", "1"), ("00", "10"), ("10", "10"), ("11", "11"),
        ("000", "110"), ("0000", "1110"), ("1111", "1111"), ("1110", "1110"), ("0111", "0111")]
for b, want in edge:
    assert max_binary_bfs(b) == want, b
    for fn in fns:
        assert fn(b) == want, (fn, b)

# đối chiếu với BFS: MỌI chuỗi nhị phân độ dài 1..11 (vét toàn bộ)
total = 0
for n in range(1, 12):
    for m in range(1 << n):
        b = format(m, f"0{n}b")
        want = max_binary_bfs(b)
        for fn in fns:
            assert fn(b) == want, (fn, b)
        total += 1
print(f"đã kiểm {total} chuỗi (mọi chuỗi độ dài ≤ 11) khớp BFS")

đã kiểm 4094 chuỗi (mọi chuỗi độ dài ≤ 11) khớp BFS


In [20]:
random.seed(2024)
for _ in range(300):
    n = random.randint(1, 3000)
    p = random.random()
    b = "".join("0" if random.random() < p else "1" for _ in range(n))
    r = max_binary_formula(b)
    assert r == max_binary_two_pointers(b) == max_binary_simulate(b)
    assert len(r) == len(b) and r.count("0") == min(1, b.count("0"))
print("n tới 3000: các cách khớp nhau; đáp án có tối đa một số 0")

n tới 3000: các cách khớp nhau; đáp án có tối đa một số 0


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Chỉ dùng phép 1 (`replace("00", "10")` lặp lại)

Bỏ qua phép 2 vì nghĩ nó "làm nhỏ chuỗi". Nhưng phép 2 là thứ gom các số 0 lại.

In [21]:
def wrong_only_op1(s):
    while "00" in s:
        i = s.find("00")
        s = s[:i] + "10" + s[i + 2:]
    return s

for b in ["000110", "0101"]:
    print(f"{b}: sai = {wrong_only_op1(b)} | đúng = {max_binary_formula(b)}")

000110: sai = 110110 | đúng = 111011
0101: sai = 0101 | đúng = 1011


**Sửa:** dùng công thức (hoặc mô phỏng có phép 2).

### Lỗi 2 — Đặt số 0 cuối cùng ở cuối chuỗi

Nghĩ "đẩy hết số 0 về cuối rồi gộp" ⇒ `"1...10"`. Nhưng phép 2 chỉ đẩy 0 sang **trái**, không sang phải.

In [22]:
def wrong_zero_at_end(s):
    if "0" not in s:
        return s
    return "1" * (len(s) - 1) + "0"

for b in ["000110", "01", "0111"]:
    print(f"{b}: sai = {wrong_zero_at_end(b)} | đúng = {max_binary_formula(b)}")

000110: sai = 111110 | đúng = 111011
01: sai = 10 | đúng = 01
0111: sai = 1110 | đúng = 0111


**Sửa:** vị trí số 0 là `k + z − 1`.

### Lỗi 3 — Quên trường hợp không có số 0

`s.find('0')` trả về `-1`; nếu không kiểm tra, công thức tạo ra chuỗi có số `0` không tồn tại.

In [23]:
def wrong_no_check(s):
    k = s.find("0"); z = s.count("0"); n = len(s)
    return "1" * (k + z - 1) + "0" + "1" * (n - k - z)

print("'111': sai =", repr(wrong_no_check("111")), "| đúng =", repr(max_binary_formula("111")))

'111': sai = '01111' | đúng = '111'


Với `"111"`: `k = −1`, `z = 0` ⇒ `"1"·(−2) + "0" + "1"·4 = "01111"` — **dài hơn** input và sai. **Sửa:** `if k == -1: return s`.

### Lỗi 4 — Tính vị trí bằng số 0 **sau** `k` mà quên trừ 1

Viết `"1" * (k + z)` (thiếu `− 1`) ⇒ chuỗi dài hơn hoặc đặt số 0 lệch phải một ô.

In [24]:
def wrong_off_by_one(s):
    k = s.find("0")
    if k == -1: return s
    z = s.count("0"); n = len(s)
    return "1" * (k + z) + "0" + "1" * (n - k - z - 1)

for b in ["000110", "01"]:
    print(f"{b}: sai = {wrong_off_by_one(b)} | đúng = {max_binary_formula(b)}")

000110: sai = 111101 | đúng = 111011
01: sai = 10 | đúng = 01


**Sửa:** với `z = 1` số 0 không di chuyển ⇒ vị trí `k = k + 1 − 1`. Luôn thử `z = 1` để kiểm tra off-by-one.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Bài "áp phép biến đổi tùy ý, tìm kết quả tối ưu" ⇒ **BFS vét cạn trên `n` nhỏ** để quan sát, rồi tìm **bất biến/đơn biến** (đại lượng không tăng/không giảm).
- Phép đổi chỗ kề `10 → 01` = "số 0 nhảy trái" — quen thuộc trong các bài sắp xếp nhị phân.
- Đáp án có cấu trúc rất đơn giản (tối đa một số 0) ⇒ chỉ cần tìm **vị trí** ⇒ đếm.
- Khi chứng minh, tìm một đại lượng `g(s)` không tăng qua mọi phép và đạt dấu bằng ở cấu hình mục tiêu.

## 15. Biến thể của bài

1. **Trả về số phép tối thiểu** để đạt chuỗi lớn nhất (khó hơn; liên quan tổng khoảng cách di chuyển của các số 0).
2. **Chỉ có phép 1** (`00 → 10`): đáp án là tham lam trái → phải (Lỗi 1 chính là lời giải của biến thể này).
3. **Tìm chuỗi nhỏ nhất** có thể đạt: phép 1 chỉ làm lớn, phép 2 đẩy 0 sang trái ⇒ dồn mọi số 0 về đầu: `"0"*z + "1"*(n−z)`… nhưng phần `1` đứng đầu thì sao? (kiểm chứng bằng BFS ở cell dưới).
4. **Phép 2 đổi thành `01 → 10`** (số 0 nhảy phải): bài khác hẳn — thử BFS để khám phá.

In [25]:
def min_binary_guess(s):
    z = s.count("0")
    return "0" * z + "1" * (len(s) - z)

ok = all(min(bfs_all(format(m, f"0{n}b"))) == min_binary_guess(format(m, f"0{n}b"))
         for n in range(1, 11) for m in range(1 << n))
print("biến thể 3 (chuỗi nhỏ nhất = dồn mọi 0 về đầu) đúng với mọi chuỗi độ dài ≤ 10:", ok)

biến thể 3 (chuỗi nhỏ nhất = dồn mọi 0 về đầu) đúng với mọi chuỗi độ dài ≤ 10: True


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. BFS | duyệt mọi chuỗi đạt được | `O(2ⁿ n²)` | `O(2ⁿ n)` | quan sát quy luật, kiểm chứng |
| 2. Mô phỏng từng phép | gom 0 về cạnh, biến 00→10 | `O(n²)` | `O(n)` | hiểu cơ chế |
| 3. Hai con trỏ | như 2, hoán đổi tắt | `O(n)` | `O(n)` | khi chưa nghĩ ra công thức |
| 4. Công thức | `k + z − 1` | `O(n)` | `O(n)` | **nộp bài** |

## 17. Bài tập (từ dễ đến khó)

**Bài 1 (đoán output):** `maximumBinaryString("100")` và `maximumBinaryString("0110")`?

**Bài 2 (tính tay):** viết dãy phép biến đổi đưa `"0101"` thành `"1011"`.

**Bài 3 (điền TODO):** hoàn thiện `last_zero_position(s)` trả về vị trí số 0 trong đáp án (hoặc `-1`).

**Bài 4 (tự viết lại):** viết lại công thức bằng **một vòng lặp** duy nhất (không dùng `find`/`count`).

**Bài 5 (mini challenge):** đếm **số chuỗi phân biệt đạt được** từ `s` (với `n ≤ 12`) bằng BFS, rồi tìm quy luật cho chuỗi dạng `"0" * z` (toàn số 0). Hãy in bảng với `z = 1..10` và đoán công thức (gợi ý: so với dãy số nổi tiếng).

In [26]:
def last_zero_position(s):
    # TODO: trả về k + z - 1 nếu có số 0, ngược lại -1
    pass

## 18. Gợi ý 3 mức

**Bài 1:** (1) `"100"`: `k = 1`, `z = 2`. (2) Vị trí `2`. (3) `"110"`; `"0110"`: `k = 0`, `z = 2` ⇒ `"1011"`.

**Bài 2:** (1) Số 0 thứ hai ở vị trí 2 nhảy trái: `0101 → 0011`. (2) `00 → 10` tại 0: `1011`. (3) Hai phép.

**Bài 3:** (1) `k = s.find('0')`. (2) `z = s.count('0')`. (3) `k + z − 1` hoặc `−1`.

**Bài 4:** (1) Duyệt từng ký tự, ghi `k` khi gặp số 0 đầu. (2) Đếm `z`. (3) Ghép chuỗi kết quả bằng `"".join` hoặc phép nhân chuỗi.

**Bài 5:** (1) Dùng `bfs_all`. (2) Với `"0"*z`, liệt kê số trạng thái. (3) Thử so với `2^{z−1}`, Fibonacci, số Catalan…

## 19. Đáp án tham khảo

In [27]:
# Bài 1
assert Solution().maximumBinaryString("100") == "110" and Solution().maximumBinaryString("0110") == "1011"

# Bài 2
steps = ["0101", "0011", "1011"]
assert all(steps[t + 1] in set(neighbors(steps[t])) for t in range(2))

# Bài 3
def last_zero_position_ans(s):
    k = s.find("0")
    return -1 if k == -1 else k + s.count("0") - 1
assert [last_zero_position_ans(x) for x in ["000110", "01", "1111", "1101001"]] == [3, 0, -1, 4]

# Bài 4
def max_binary_one_loop(s):
    k = -1; z = 0
    for i, ch in enumerate(s):
        if ch == "0":
            if k == -1:
                k = i
            z += 1
    if k == -1:
        return s
    pos = k + z - 1
    return "1" * pos + "0" + "1" * (len(s) - pos - 1)

for n in range(1, 11):
    for m in range(1 << n):
        b = format(m, f"0{n}b")
        assert max_binary_one_loop(b) == max_binary_formula(b)
print("Bài 1-4: OK")

Bài 1-4: OK


In [28]:
# Bài 5 — số trạng thái đạt được từ "0"*z
rows = []
for z in range(1, 11):
    rows.append(len(bfs_all("0" * z)))
print("z:          ", list(range(1, 11)))
print("số trạng thái:", rows)
print("2^z - 1:      ", [2 ** z - 1 for z in range(1, 11)])
assert rows == [2 ** z - 1 for z in range(1, 11)]

z:           [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
số trạng thái: [1, 3, 7, 15, 31, 63, 127, 255, 511, 1023]
2^z - 1:       [1, 3, 7, 15, 31, 63, 127, 255, 511, 1023]


Quan sát (đã kiểm bằng `assert` với `z ≤ 10`): từ `"0"·z` đạt được đúng `2^z − 1` chuỗi. Giải thích: mọi chuỗi đạt được đều còn ít nhất một số 0 (nên `"1"·z` không đạt được), và ngược lại — như cell dưới kiểm chứng — **mọi** chuỗi độ dài `z` có ít nhất một số 0 đều đạt được; có đúng `2^z − 1` chuỗi như vậy. Điều này cho thấy số trạng thái tăng theo hàm mũ, giải thích vì sao BFS không dùng được cho `n` lớn.

Thêm một kiểm chứng cho nhận định "mọi chuỗi khác `1…1` đều đạt được":

In [29]:
for z in range(1, 11):
    reach = bfs_all("0" * z)
    assert reach == {format(m, f"0{z}b") for m in range(2 ** z - 1)}
print("đúng: từ '0'*z đạt được mọi chuỗi độ dài z trừ '1'*z (z ≤ 10)")

đúng: từ '0'*z đạt được mọi chuỗi độ dài z trừ '1'*z (z ≤ 10)


## 20. Template nhớ nhanh + Checklist

```python
class Solution:
    def maximumBinaryString(self, binary: str) -> str:
        k = binary.find("0")
        if k == -1:
            return binary                       # không có số 0
        z = binary.count("0")
        n = len(binary)
        return "1" * (k + z - 1) + "0" + "1" * (n - k - z)
```

**Checklist trước khi submit:**
- [ ] Trường hợp không có số 0 ⇒ trả về nguyên chuỗi.
- [ ] Đáp án dài đúng `n`, chứa **đúng một** số 0 (khi `z ≥ 1`).
- [ ] Vị trí số 0: `k + z − 1` (thử `z = 1`: không đổi).
- [ ] Phần `1` đứng đầu chuỗi gốc được giữ nguyên.
- [ ] Không dùng `replace` lặp `O(n²)` với `n = 10⁵`.